# 04 · STAR on Euler (Tutorial 2-2-1)
Same 27 runs, same fastp trimming and same GENCODE vM25 annotation as the kallisto pipeline, so notebook 05 can compare the **quantification method** alone.

| Step | Where | Script | Resources |
|---|---|---|---|
| conda env | login node | `euler/00_setup.sh` | – |
| genome, annotation, 27 `.sra` | login node (tmux) | `euler/01_download.sh` | ~55 GB scratch |
| STAR genome index | SLURM | `euler/02_star_index.sh` | 10 cores × 6 GB (~30 GB needed), ~1–2 h |
| fastp + STAR per run | SLURM array | `euler/03_map_array.sh` | 8 cores × 6 GB, ~30–60 min each, 6 at a time |
| pack results | login node | `euler/04_collect.sh` | a few MB |

`--quantMode GeneCounts TranscriptomeSAM`: per-gene counts feed DESeq2; the transcriptome BAM is kept for optional RSEM.

## On Euler (ETH network or VPN)
```bash
ssh <user>@euler.ethz.ch
git clone https://github.com/Jeffateth/RNAseq_aging && bash RNAseq_aging/euler/00_setup.sh   # once
tmux new -s dl && bash RNAseq_aging/euler/01_download.sh                                  # resumable; detach with Ctrl-b d
bash RNAseq_aging/euler/submit.sh        # index job, then the 27-run mapping array (starts when the index is done)
squeue -u $USER                          # monitor; logs in $SCRATCH/rnaseq_aging/logs
bash RNAseq_aging/euler/04_collect.sh    # when all runs are finished -> $SCRATCH/rnaseq_aging/star_results.tar.gz
```
Copy the result to this repo (from the laptop):
```bash
mkdir -p results/star && scp <user>@euler.ethz.ch:/cluster/scratch/<user>/rnaseq_aging/star_results.tar.gz results/star/
```
Scratch files are deleted after 15 days: copy results back promptly.

## On the laptop: load the results

In [ ]:
import sys, tarfile; sys.path.insert(0, '../scripts')
import helpers as h
runs = h.load_runs()
with tarfile.open(h.STAR / 'star_results.tar.gz') as t:
    t.extractall(h.STAR, filter='data')
counts, qc = h.load_star(runs)
print(counts.shape, 'genes x runs')
qc

Checks: `unique_%` should be high (≈ 80–90 %), and `frac_read1_same_strand` ≈ 0.5 confirms the libraries are unstranded (the counts used are the unstranded column).

In [ ]:
assert counts.shape[1] == 27
assert qc.frac_read1_same_strand.between(0.4, 0.6).all(), 'library looks stranded: use another ReadsPerGene column'
counts.to_csv(h.STAR / 'counts_unstranded.csv.gz')
qc.to_csv(h.STAR / 'star_qc.csv')
print('saved results/star/counts_unstranded.csv.gz')

## Mapping rates: STAR vs kallisto

In [ ]:
k = h.qc_summary(runs)[['aligned_%']].rename(columns={'aligned_%': 'kallisto_pseudoaligned_%'})
qc[['unique_%', 'multi_%', 'assigned_to_gene_%']].join(k).join(runs['group'])